# Analysis Notebook for Keithley CLI data

This notebook is used for analyzing the measurement data obtained from the Keithley CLI. It includes functions to download and combine `.txt` files from a Nextcloud share into a single DataFrame, as well as cells showing how to visualize the data using Plotly Express.

In [12]:
## Importing Libraries and Helper Functions
import plotly.express as px
import pandas as pd
from helpers.nextcloud import nextcloud_txt_to_dataframe, nextcloud_csv_to_dataframe

In [13]:
## Download and combine .txt files into a DataFrame, then print row counts per source file.
df = nextcloud_txt_to_dataframe()


print("RADEF txt files:")
for file in df["source_file"].unique() if not df.empty else []:
    subset = df[df["source_file"] == file]
    print(f"File: {file}, Data Rows: {len(subset)}")

if df.empty:
    print("No data loaded from the Nextcloud share.")


RADEF txt files:
File: raw_data/dev3_idvg.txt, Data Rows: 51
File: raw_data/gq_test2_short_cable_1nA_gq.txt, Data Rows: 1966
File: raw_data/gq_test3_short_cable_1nA_1V_gq.txt, Data Rows: 1276
File: raw_data/gq_test3_short_cable_1nA_gq.txt, Data Rows: 1385
File: raw_data/gq_test5_gq.txt, Data Rows: 2684
File: raw_data/gq_test6_gq.txt, Data Rows: 145
File: raw_data/gq_test_100nA_gq.txt, Data Rows: 14
File: raw_data/gq_test_20nA_gq.txt, Data Rows: 72
File: raw_data/gq_test_dev2_1nA_gq.txt, Data Rows: 691
File: raw_data/gq_test_dev2_20nA_gq.txt, Data Rows: 481
File: raw_data/gq_test_dev2_2nA_gq.txt, Data Rows: 700
File: raw_data/gq_test_short_cable_1nA_1V_cable_test2_gq.txt, Data Rows: 1384
File: raw_data/gq_test_short_cable_1nA_1V_cable_test3_gq.txt, Data Rows: 1386
File: raw_data/gq_test_short_cable_1nA_1V_cable_test_gq.txt, Data Rows: 2643
File: raw_data/gq_test_short_cable_1nA_1V_dev3_gq.txt, Data Rows: 1363
File: raw_data/gq_test_short_cable_1nA_gq.txt, Data Rows: 489
File: raw_data/g

## Gate Charge Data

In [14]:
# Get gate charge data and plot gate voltage vs. gate charge for files containing "_gq" in their name.

gate_charge_df = df[df["source_file"].str.contains("_gq", case=False, na=False)]

if gate_charge_df.empty:
    print("No gate charge data found in the DataFrame.")
else:
    # Integrate gate current over time to calculate gate charge (Q = \int_0^t I(t) * dt).
    dt = gate_charge_df["Time"].diff().fillna(0)  # Time intervals
    gate_charge_df["Gate Charge"] = (
        gate_charge_df["Gate Current"] * dt
    ).cumsum()  # Cumulative sum to get total charge

    fig = px.line(
        gate_charge_df, x="Gate Charge", y="Gate Voltage", color="source_file"
    )

    fig.update_layout(
        width=800,
        height=600,
        xaxis_range=[0, 50e-9],
    )
    fig.show()

## ID-VD Data

In [15]:
# Get ID-VD data and plot Drain Current and Gate Current vs. Drain Voltage on the same graph, with separate lines for each source file.
idvd_df = df[df["source_file"].str.contains("_idvd", case=False, na=False)]
if idvd_df.empty:
    print("No ID-VD data found in the DataFrame.")
else:
    fig_idvd = px.line(
        idvd_df,
        x="Drain Voltage",
        y=["Drain Current", "Gate Current"],
        color="source_file",
        title="ID-VD Characteristics",
        labels={"value": "Current (Amps)", "variable": "Current Type"},
    )
    fig_idvd.update_layout(
        xaxis_title="Drain Voltage (Volts)",
        yaxis_title="Current (Amps)",
        legend_title="Source File",
        width=800,
        height=600,
    )
    fig_idvd.show()

No ID-VD data found in the DataFrame.


## ID-VG Data

In [16]:
# Get ID-VG data and plot Drain Current and Gate Current vs. Gate Voltage on the same graph, with separate lines for each source file.
idvg_df = df[df["source_file"].str.contains("_idvg", case=False, na=False)]

if idvg_df.empty:
    print("No ID-VG data found in the DataFrame.")
else:
    fig_idvg = px.line(
        idvg_df,
        x="Gate Voltage",
        y=["Drain Current", "Gate Current"],
        color="source_file",
        markers=True,
        symbol="variable",
    )

    fig_idvg.update_layout(
        width=800,
        height=600,
        yaxis_type="log",  # Logarithmic scale for current to better visualize low currents
    )
    fig_idvg.show()


### Sanity Check Between Vanderbilt and JYU Data

In [19]:
df_joshua = nextcloud_csv_to_dataframe()

print("Joshua's CSV files:")
for file in df_joshua["source_file"].unique() if not df_joshua.empty else []:
    subset = df_joshua[df_joshua["source_file"] == file]
    print(f"File: {file}, Data Rows: {len(subset)}")

idvg_df_joshua = df_joshua[
    df_joshua["source_file"].str.contains("_idvg", case=False, na=False)
]
# Remove DataName column if it exists, as it may interfere with plotting
idvg_df_joshua = idvg_df_joshua.drop(columns=["DataName"], errors="ignore")

# Rename joshua's ID-VG columns to match RADEF's for easier plotting
col_mapping = {
    "Vgate": "Gate Voltage",
    "Isource": "Source Voltage",
    "Idrain": "Drain Current",
    "Igate": "Gate Current",
}

# strip whitespace from column names in joshua's DataFrame before renaming
idvg_df_joshua.columns = idvg_df_joshua.columns.str.strip()

idvg_df_joshua = idvg_df_joshua.rename(columns=col_mapping)

# Combine both dataframes
idvg_combined = pd.concat([idvg_df, idvg_df_joshua], ignore_index=True)

fig_combined = px.line(
    idvg_combined,
    x="Gate Voltage",
    y=["Drain Current", "Gate Current"],
    color="source_file",
    symbol="variable",
    title="Combined ID-VG Characteristics (RADEF and Joshua's Data)",
)

fig_combined.update_layout(
    width=800,
    height=600,
    yaxis_type="log",  # Logarithmic scale for current to better visualize low currents
)
fig_combined.show()


Joshua's CSV files:
File: raw_data/A010_3_4_2026 9_19_32 AM_idvg.csv, Data Rows: 126


## Irradiation Data

In [18]:
## Irradiation Data
irrad_df = df[df["source_file"].str.contains("_irrad", case=False)]
if irrad_df.empty:
    print("No irradiation data found in the DataFrame.")
else:
    fig_irrad = px.line(
        irrad_df,
        x="Fluence",
        y=["Drain Current", "Gate Current"],
        color="source_file",
        title="Irradiation Test Data",
    )

    fig_irrad.update_layout(
        width=800,
        height=600,
    )
    fig_irrad.show()

No irradiation data found in the DataFrame.
